In [ ]:
'''
This is a notebook to help write config files for different arg inference tools
In cases where a large contig has to be broken down into multiple differing runs
This will output a csv file with the params file for each tool
'''

In [ ]:
import yaml
import pandas as pd
import re
import glob
import numpy as np
import os

pd.set_option('display.max_colwidth', None)

In [ ]:
#DEFINE Defaults:
MAX_CONTIG_LENGTH = 5_000_000
CONTIG_OVERLAP = 300_000
MIN_CONTIG_LENGTH = 750_000

NE = 10_000
SINGULARITY_IMAGE = "/gpfs/home/shkhalid/arg_inference_tools.sif"
OUTPUT_DIR = "/gpfs/home/shkhalid/SiepelVeeramahSharedDrive/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/singer"
CONFIG_OUTPUT = "/gpfs/home/shkhalid/SiepelVeeramahSharedDrive/argsortium-inference/singer/myconfigs/2026_05_11_runs"
input_folder_base = "/gpfs/home/shkhalid/SiepelVeeramahSharedDrive/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09"


patterns = [
    f"{input_folder_base}/CEU_*_CHB_20_YRI_20/*.params.csv",
    f"{input_folder_base}/CEU_*_CHB_20_YRI_0/*.params.csv",
    f"{input_folder_base}/CEU_*_CHB_0_YRI_20/*.params.csv",
    f"{input_folder_base}/CEU_*_CHB_125_YRI_125/*.params.csv"
]

arg_inference_method = "singer".lower()

In [ ]:
def sort_samples_str(samples_str):
    '''
    This function will be deprecated once the naming convention in the simulations has been updated.
    '''
    # Split into (pop, count) pairs: ["YRI", "20", "CEU", "0", "CHB", "0"]
    parts = samples_str.split("_")
    # Zip into pairs: [("YRI", "20"), ("CEU", "0"), ("CHB", "0")]
    pairs = [(parts[i], parts[i+1]) for i in range(0, len(parts), 2)]
    # Sort alphabetically by pop name and rejoin
    return "_".join([f"{pop}_{count}" for pop, count in sorted(pairs)])

def chunk_row(row):
    """Break a row into overlapping chunks if simulated_length > MAX_CONTIG_LENGTH.
    Pad chunks smaller than MIN_CONTIG_LENGTH by extending left or right."""
    if row["simulated_length"] <= MAX_CONTIG_LENGTH:
        chunks = [row.copy()]
        chunks[0]["inference_start"] = row["start"]
        chunks[0]["inference_end"] = row["end"]
        chunks[0]["inference_length"] = row["end"] - row["start"]
    else:
        step = MAX_CONTIG_LENGTH - CONTIG_OVERLAP
        chunks = []
        chunk_start = row["start"]

        while chunk_start < row["end"]:
            chunk_end = min(chunk_start + MAX_CONTIG_LENGTH, row["end"])
            new_row = row.copy()
            new_row["inference_start"] = chunk_start
            new_row["inference_end"] = chunk_end
            new_row["inference_length"] = chunk_end - chunk_start
            chunks.append(new_row)

            if chunk_end == row["end"]:
                break
            chunk_start += step

    # Pad any chunk below MIN_CONTIG_LENGTH
    for chunk in chunks:
        if chunk["inference_length"] < MIN_CONTIG_LENGTH:
            deficit = MIN_CONTIG_LENGTH - chunk["inference_length"]
            if chunk["inference_end"] == row["end"]:
                # At the end of the region — extend leftward
                chunk["inference_start"] = max(row["start"], chunk["inference_start"] - deficit)
            else:
                # At the start of the region — extend rightward
                chunk["inference_end"] = min(row["end"], chunk["inference_end"] + deficit)
            chunk["inference_length"] = chunk["inference_end"] - chunk["inference_start"]

    return chunks

In [ ]:
#define all configs here
argweaver_config = {
    "singularity": SINGULARITY_IMAGE,
    "output_dir": "",
    "params_pattern": [],
    "max_contig_length": MAX_CONTIG_LENGTH,
    "start": None,
    "end": None,
    "Ne": None,
    "recomb_rate": None,
    "mcmc_samples": 4000, #defaults
    "compression": 10,
    "n_time_points": None,
    "sample_step": 20,
}

singer_config = {
    "singularity" : SINGULARITY_IMAGE,
    "output_dir" : "",
    "params_pattern" : [],
    "max_contig_length" : MAX_CONTIG_LENGTH,
    "start" : None,
    "end" : None,
    "Ne" : 2*10_000, #haploid Ne
    "mcmc_samples" : 2000, #SINGER converges quicker
    "recomb_ratio" : None,
    "thin" : 20,
    "polar" : 0.99 #use 0.99 for polarized data
    
}

relate_config = {
    
}

tsinfer_config = {
    
}

threads_config = {
    
}

asmc_clust_config = {
    
}

polegon_config = {
    
}

config_map = {
    "argweaver" : argweaver_config,
    "singer" : singer_config,
    "relate" : relate_config,
    "tsinfer" : tsinfer_config,
    "asmc_clust" : asmc_clust_config,
    "polegon_config" : polegon_config
}

config_to_use = config_map[arg_inference_method]

In [ ]:
#find all params files and read them in to 1 pandas dataframe:
matched_params_files = sorted(
    f for pattern in patterns for f in glob.glob(pattern)
)

params_df = pd.concat(
    [pd.read_csv(f).assign(source_file=f) for f in matched_params_files],
    ignore_index=True
)
params_df["samples"] = params_df["samples"].apply(sort_samples_str)

In [ ]:
grouped_df = params_df.groupby(["samples", "contig", "start", "end", "simulated_length"]).count()[["vcf_file"]]\
.rename(columns = {"vcf_file" : "number_of_seeds"}).reset_index()

chunked_df = pd.DataFrame(
    [chunk for row in grouped_df.itertuples(index=False)
     for chunk in chunk_row(row._asdict())]
).reset_index(drop=True)

chunked_df

In [ ]:
merged_file = pd.merge(chunked_df, params_df, on = ["contig", "samples", "start", "end", "simulated_length"])
merged_file["output_dir"] = merged_file[["samples", "contig", "start", "end"]]\
.apply(lambda x : "/".join([OUTPUT_DIR, str(x[0]), "_".join([str(x[1]), str(x[2]), str(x[3])])]), axis = 1)

In [ ]:
merged_file

In [ ]:
import csv as csv_module

MASTER_CSV_PATH = os.path.join(CONFIG_OUTPUT, "master_params.csv")
os.makedirs(CONFIG_OUTPUT, exist_ok=True)

output_rows = []
seen_uids = set()

for _, row in merged_file.iterrows():
    vcf_basename = os.path.basename(row["vcf_file"]).replace(".no_multiallelics.filtered.vcf.gz", "")
    pop_str = row["samples"]
    inf_start = int(row["inference_start"])
    inf_end   = int(row["inference_end"])

    # Append chunk window to uid when the inference window is a sub-region of the simulated contig
    if inf_start != int(row["start"]) or inf_end != int(row["end"]):
        uid = f"{pop_str}__{vcf_basename}__{inf_start}_{inf_end}"
    else:
        uid = f"{pop_str}__{vcf_basename}"

    assert uid not in seen_uids, f"Duplicate uid: {uid}"
    seen_uids.add(uid)

    output_rows.append({
        "uid":             uid,
        "vcf_file":        row["vcf_file"],
        "mu":              row["mu"],
        "recomb_map":      row["recomb_map"],
        "output_dir":      row["output_dir"],
        "inference_start": inf_start,
        "inference_end":   inf_end,
    })

fieldnames = ["uid", "vcf_file", "mu", "recomb_map", "output_dir", "inference_start", "inference_end"]
with open(MASTER_CSV_PATH, "w", newline="") as f:
    writer = csv_module.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(output_rows)

print(f"Wrote {len(output_rows)} runs to {MASTER_CSV_PATH}")

In [ ]:
pd.read_csv("/gpfs/home/shkhalid/SiepelVeeramahSharedDrive/argsortium-inference/singer/myconfigs/2026_05_11_runs/master_params.csv")